# Token sind keine Wörter

Ein Sprachmodell bekommt keinen Text, sondern eine Folge von Ganzzahlen. Die Übersetzung macht der Tokenizer, mit einem festen Vokabular, das zum Modell gehört. Hier probiert ihr dieselben zwei Tokenizer aus wie auf der Folie: Qwen2.5 und GPT-2. Geladen werden nur die Tokenizer, kein Modell.

1. Tokenizer laden
2. Die Beispiele der Folie
3. Eine eigene Java-Zeile
4. Derselbe Satz auf Deutsch und Englisch
5. Eine lange Zahl
6. Buchstaben zählen

**Pflicht**, zu zweit in 10 Minuten: Schritte 1 bis 5. Zellen der Reihe nach ausführen (Shift+Enter), in 3 bis 5 eigene Eingaben ausprobieren.

**Zusatz** für Schnelle: Schritt 6.

Die Lösungen stehen zugeklappt direkt nach der Aufgabe.

## Pflicht

### 1. Tokenizer laden

Der Tokenizer gehört zum Modell, deshalb lädt man ihn unter dem Namen des Modells. Die beiden Tokenizer sind zusammen rund 14 MB gross. Eine Warnung zu `HF_TOKEN` könnt ihr ignorieren, es geht ohne Konto.

In [ ]:
# Nur in Colab: Hilfsdateien und Daten aus dem Repo holen. Lokal tut die Zelle nichts.
import os
import subprocess
import sys

if "google.colab" in sys.modules:
    repo = "/content/ki-workshop-handson"
    if not os.path.exists(repo):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/dprinz/ki-workshop-handson.git", repo], check=True)
    os.chdir(f"{repo}/tokenizer")

In [1]:
from transformers import AutoTokenizer

qwen = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B")
gpt2 = AutoTokenizer.from_pretrained("openai-community/gpt2")

print(f"Qwen2.5 kennt {len(qwen):_} Token, GPT-2 {len(gpt2):_}".replace("_", "'"))

Qwen2.5 kennt 151'665 Token, GPT-2 50'257


Wie auf der Folie ein Codec: `encode` macht aus Text ein Array von Ganzzahlen, `decode` macht daraus wieder Text.

In [2]:
ids = qwen.encode("Der Build ist fehlgeschlagen, weil ein Test rot ist.")
print(ids)
print(qwen.decode(ids))

[22171, 7854, 5999, 1152, 17958, 45061, 54708, 11, 52507, 4368, 3393, 5749, 5999, 13]
Der Build ist fehlgeschlagen, weil ein Test rot ist.


### 2. Die Beispiele der Folie

`zeige` zerlegt einen Text und zeigt die Token abwechselnd hinterlegt, Leerzeichen als ·, darunter die IDs. Die Zahlen kennt ihr von der Folie.

In [3]:
from helfer import zeige

zeige("Der Build ist fehlgeschlagen, weil ein Test rot ist.", qwen)
zeige("The build failed because a test is red.", qwen)
zeige("private static final int CHARS_PER_TOKEN = 4;", qwen)
zeige("Thurgauer Kantonalbank", qwen)
zeige(" strawberry", qwen)

Qwen2.5: 14 Token
Der|·Build|·ist|·fe|hl|gesch|lagen|,|·weil|·ein|·Test|·rot|·ist|.
[22171, 7854, 5999, 1152, 17958, 45061, 54708, 11, 52507, 4368, 3393, 5749, 5999, 13]

Qwen2.5: 9 Token
The|·build|·failed|·because|·a|·test|·is|·red|.
[785, 1936, 4641, 1576, 264, 1273, 374, 2518, 13]

Qwen2.5: 12 Token
private|·static|·final|·int|·CH|ARS|_PER|_TOKEN|·=|·|4|;
[1996, 1099, 1590, 526, 6826, 17048, 10617, 18681, 284, 220, 19, 26]

Qwen2.5: 6 Token
Th|urg|auer|·Kant|onal|bank
[1001, 5556, 27097, 62162, 24202, 17033]

Qwen2.5: 1 Token
·strawberry
[72600]

Derselbe deutsche Satz mit beiden Tokenizern. Wo zerlegt GPT-2 anders?

In [4]:
zeige("Der Build ist fehlgeschlagen, weil ein Test rot ist.", qwen, gpt2)

Qwen2.5: 14 Token
Der|·Build|·ist|·fe|hl|gesch|lagen|,|·weil|·ein|·Test|·rot|·ist|.
[22171, 7854, 5999, 1152, 17958, 45061, 54708, 11, 52507, 4368, 3393, 5749, 5999, 13]

GPT-2: 20 Token
Der|·Build|·is|t|·fe|hl|ges|ch|l|agen|,|·we|il|·e|in|·Test|·rot|·is|t|.
[28532, 10934, 318, 83, 730, 18519, 3212, 354, 75, 11286, 11, 356, 346, 304, 259, 6208, 5724, 318, 83, 13]

### 3. Eine eigene Java-Zeile

Nehmt eine Zeile aus eurem Code, gern mit Einrückung, Annotation und langen Bezeichnern. Was ist ein einziges Token, was wird zerlegt?

In [5]:
zeile = "        @Autowired private KundenRepository kundenRepository;"
zeige(zeile, qwen, gpt2)

Qwen2.5: 10 Token
·······|·@|Autowired|·private|·Kunden|Repository|·k|unden|Repository|;
[286, 569, 13192, 869, 61291, 4624, 595, 22157, 4624, 26]

GPT-2: 22 Token
·|·|·|·|·|·|·|·@|Aut|ow|ired|·private|·Kund|en|Rep|ository|·k|und|en|Rep|ository|;
[220, 220, 220, 220, 220, 220, 220, 2488, 16541, 322, 1202, 2839, 45099, 268, 6207, 13264, 479, 917, 268, 6207, 13264, 26]

In [6]:
# @title Lösung (aufklappen)
# Häufige Schlüsselwörter wie private, static, final oder int sind je ein Token, mit dem
# Leerzeichen davor. Eigene Bezeichner werden zerlegt, oft an camelCase und Unterstrichen.
# Derselbe Wortteil hat je nach Leerzeichen und Grossschreibung eine andere ID:
# " Kunden" ist bei Qwen2.5 ein Token, in "kundenRepository" wird daraus k|unden.
# Einrückung: Qwen2.5 fasst die Leerzeichen zu einem Token zusammen, GPT-2 braucht für
# jedes ein eigenes. Tief eingerückter Code kostet bei GPT-2 deutlich mehr Token:
beispiel_java = "                return kundenRepository.findById(kundenId).orElseThrow();"
zeige(beispiel_java, qwen, gpt2)

Qwen2.5: 13 Token
···············|·return|·k|unden|Repository|.findById|(k|unden|Id|).|orElse|Throw|();
[394, 470, 595, 22157, 4624, 21648, 5969, 22157, 764, 568, 54088, 23079, 2129]

GPT-2: 34 Token
·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·return|·k|und|en|Rep|ository|.|find|ById|(|k|und|en|Id|).|or|Else|Throw|();
[220, 220, 220, 220, 220, 220, 220, 220, 220, 220, 220, 220, 220, 220, 220, 1441, 479, 917, 268, 6207, 13264, 13, 19796, 48364, 7, 74, 917, 268, 7390, 737, 273, 40674, 39431, 9783]

### 4. Derselbe Satz auf Deutsch und Englisch

Schreibt einen Satz auf Deutsch und seine englische Übersetzung. Wie viele Token mehr braucht Deutsch, bei Qwen2.5 und bei GPT-2? Was heisst das für Kontextfenster und Preise?

In [7]:
deutsch = "Der Kunde hat seine Adresse geändert."
englisch = "The customer has changed their address."
zeige(deutsch, qwen, gpt2)
zeige(englisch, qwen, gpt2)

Qwen2.5: 10 Token
Der|·K|unde|·hat|·seine|·Adresse|·ge|änd|ert|.
[22171, 730, 28865, 8896, 38897, 89470, 3893, 21545, 529, 13]

GPT-2: 14 Token
Der|·Kund|e|·hat|·se|ine|·Ad|res|se|·ge|ä|nder|t|.
[28532, 45099, 68, 6877, 384, 500, 1215, 411, 325, 4903, 11033, 681, 83, 13]

Qwen2.5: 7 Token
The|·customer|·has|·changed|·their|·address|.
[785, 6002, 702, 5497, 862, 2621, 13]

GPT-2: 7 Token
The|·customer|·has|·changed|·their|·address|.
[464, 6491, 468, 3421, 511, 2209, 13]

In [8]:
# @title Lösung (aufklappen)
# Deutsch braucht mehr Token, weil Tokenizer mit viel englischem Text gebaut werden und
# deutsche Wörter deshalb öfter zerlegt werden (Petrov et al. 2023). Bei GPT-2 ist der
# Unterschied grösser: Seine Trainingsdaten waren fast nur englisch (Radford et al. 2019).
# Kontextfenster, Limits und Preise zählen in Token. Derselbe Inhalt auf Deutsch füllt
# das Fenster schneller und kostet mehr. Die Faustregel von vier Zeichen pro Token gilt
# für Englisch.
paare = [
    ("Der Build ist fehlgeschlagen, weil ein Test rot ist.", "The build failed because a test is red."),
    ("Die Pipeline ist grün, aber der Deploy auf Produktion ist fehlgeschlagen.",
     "The pipeline is green, but the deployment to production failed."),
]
for de, en in paare:
    print(de, "/", en)
    for name, tokenizer in [("Qwen2.5", qwen), ("GPT-2", gpt2)]:
        n_de, n_en = len(tokenizer.encode(de)), len(tokenizer.encode(en))
        print(f"  {name:8} Deutsch {n_de:2} Token ({len(de) / n_de:.1f} Zeichen pro Token), "
              f"Englisch {n_en:2} ({len(en) / n_en:.1f}), Faktor {n_de / n_en:.1f}")

Der Build ist fehlgeschlagen, weil ein Test rot ist. / The build failed because a test is red.
  Qwen2.5  Deutsch 14 Token (3.7 Zeichen pro Token), Englisch  9 (4.3), Faktor 1.6
  GPT-2    Deutsch 20 Token (2.6 Zeichen pro Token), Englisch  9 (4.3), Faktor 2.2
Die Pipeline ist grün, aber der Deploy auf Produktion ist fehlgeschlagen. / The pipeline is green, but the deployment to production failed.
  Qwen2.5  Deutsch 18 Token (4.1 Zeichen pro Token), Englisch 12 (5.2), Faktor 1.5
  GPT-2    Deutsch 25 Token (2.9 Zeichen pro Token), Englisch 12 (5.2), Faktor 2.1


### 5. Eine lange Zahl

Nehmt eine lange Zahl, etwa `Long.MAX_VALUE`, einen Zeitstempel in Millisekunden oder einen Betrag. Wie zerlegen die beiden Tokenizer sie? Was heisst das, wenn ein Modell damit rechnen soll?

In [9]:
zahl = "Long.MAX_VALUE = 9223372036854775807"
zeige(zahl, qwen, gpt2)

Qwen2.5: 24 Token
Long|.MAX|_VALUE|·=|·|9|2|2|3|3|7|2|0|3|6|8|5|4|7|7|5|8|0|7
[6583, 16596, 7476, 284, 220, 24, 17, 17, 18, 18, 22, 17, 15, 18, 21, 23, 20, 19, 22, 22, 20, 23, 15, 22]

GPT-2: 15 Token
Long|.|MAX|_|VALUE|·=|·9|22|33|720|36|85|47|75|807
[14617, 13, 22921, 62, 39488, 796, 860, 1828, 2091, 23906, 2623, 5332, 2857, 2425, 36928]

In [10]:
# @title Lösung (aufklappen)
# Qwen2.5 zerlegt jede Zahl in einzelne Ziffern, das legt eine Regel im Tokenizer fest.
# So machen es auch PaLM und Llama (Chowdhery et al. 2022, Touvron et al. 2023).
# GPT-2 zerlegt in Stücke, die in seinen Trainingsdaten häufig waren, nicht nach Stellen:
# 1000000 wird zu 1|000000, eins mehr zu 10000|01.
# Das Modell sieht also Token-IDs und keine Zahlen. Exakt rechnen fällt ihm schwer,
# ein Tool rechnet sicher.
for beispiel_zahl in ["1000000", "1000001"]:
    zeige(beispiel_zahl, qwen, gpt2)
print(1000000 + 1)

Qwen2.5: 7 Token
1|0|0|0|0|0|0
[16, 15, 15, 15, 15, 15, 15]

GPT-2: 2 Token
1|000000
[16, 10535]

Qwen2.5: 7 Token
1|0|0|0|0|0|1
[16, 15, 15, 15, 15, 15, 16]

GPT-2: 2 Token
10000|01
[49388, 486]

1000001


## Zusatz

### 6. Buchstaben zählen

Wie viele r stecken in „strawberry“? Ihr seht es sofort, ein Modell bekommt nur die IDs. Zerlegt das Wort mit und ohne Leerzeichen davor, gross geschrieben und Buchstabe für Buchstabe. Was sieht das Modell jeweils? Warum hilft es, wenn ein Modell das Wort zuerst buchstabiert?

In [11]:
for wort in [" strawberry", "strawberry", "Strawberry", "s t r a w b e r r y"]:
    zeige(wort, qwen)

Qwen2.5: 1 Token
·strawberry
[72600]

Qwen2.5: 3 Token
str|aw|berry
[495, 672, 15357]

Qwen2.5: 3 Token
Str|aw|berry
[2580, 672, 15357]

Qwen2.5: 10 Token
s|·t|·r|·a|·w|·b|·e|·r|·r|·y
[82, 259, 435, 264, 289, 293, 384, 435, 435, 379]

In [12]:
# @title Lösung (aufklappen)
# " strawberry" ist bei beiden Tokenizern ein einziges Token. Das Modell bekommt eine Zahl,
# nicht die zehn Buchstaben darin. Wie viele r in dieser ID stecken, steht nirgends in der
# Eingabe; das muss das Modell aus dem Training mitbringen.
# Ohne Leerzeichen oder gross geschrieben sind es andere IDs und andere Stücke.
# Buchstabiert ist jeder Buchstabe ein eigenes Token, die drei r haben dieselbe ID. Modelle,
# die vor der Antwort Zwischenschritte schreiben, zählen deshalb oft richtig: Sie schreiben
# das Wort zuerst Buchstabe für Buchstabe hin. Sicher ist ein Tool.
print(qwen.encode(" strawberry"), "gegen", qwen.encode("s t r a w b e r r y"))
print("strawberry".count("r"))

[72600] gegen [82, 259, 435, 264, 289, 293, 384, 435, 435, 379]
3


Darauf kommen wir bei den Grenzen eines LLM zurück.